# CS3081 — Uncertainty and Bayesian Networks
## Exercise 5(c) — Google Colab
Run all cells in order. The first code cell installs pgmpy.


In [3]:
%pip -q install pgmpy


In [4]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD, State
from pgmpy.inference import VariableElimination
from pgmpy.sampling import BayesianModelSampling
import numpy as np

np.random.seed(42)

In [5]:
model = DiscreteBayesianNetwork([
    ("heatwave", "overload"),
    ("heatwave", "powercut"),
    ("overload", "powercut"),
    ("powercut", "cancel")
])

cpd_h = TabularCPD("heatwave", 2, [[0.3], [0.7]],
                   state_names={"heatwave": ["yes", "no"]})
cpd_a = TabularCPD("overload", 2, [[0.6, 0.1], [0.4, 0.9]],
                   evidence=["heatwave"], evidence_card=[2],
                   state_names={"overload": ["yes", "no"], "heatwave": ["yes", "no"]})
cpd_c = TabularCPD("powercut", 2,
                   [[0.7, 0.2, 0.5, 0.05], [0.3, 0.8, 0.5, 0.95]],
                   evidence=["heatwave", "overload"], evidence_card=[2, 2],
                   state_names={"powercut": ["yes", "no"],
                                "heatwave": ["yes", "no"],
                                "overload": ["yes", "no"]})
cpd_l = TabularCPD("cancel", 2, [[0.8, 0.1], [0.2, 0.9]],
                   evidence=["powercut"], evidence_card=[2],
                   state_names={"cancel": ["yes", "no"], "powercut": ["yes", "no"]})

model.add_cpds(cpd_h, cpd_a, cpd_c, cpd_l)
print("Model valid:", model.check_model())

Model valid: True


In [6]:
infer = VariableElimination(model)
posterior = infer.query(variables=["heatwave"], evidence={"cancel": "yes"})
print(posterior)
exact = float(posterior.get_value(heatwave="yes"))
print(f"Exact P(H=yes | L=yes) = {exact:.4f}")

+---------------+-----------------+
| heatwave      |   phi(heatwave) |
+===============+=================+
| heatwave(yes) |          0.5367 |
+---------------+-----------------+
| heatwave(no)  |          0.4633 |
+---------------+-----------------+
Exact P(H=yes | L=yes) = 0.5367


In [7]:
sampler = BayesianModelSampling(model)
samples = sampler.rejection_sample(
    evidence=[State("cancel", "yes")], size=10000, show_progress=False
)
print(samples.head())
estimate = (samples["heatwave"] == "yes").mean()
print(f"Rejection sampling estimate (n=10000) = {estimate:.4f}")
print(f"Absolute difference = {abs(exact-estimate):.4f}")

  heatwave overload powercut cancel
0      yes      yes      yes    yes
1      yes      yes      yes    yes
2      yes       no      yes    yes
3       no       no       no    yes
4      yes      yes      yes    yes
Rejection sampling estimate (n=10000) = 0.5430
Absolute difference = 0.0063


**Comparison:** Exact inference gives P(H=yes | L=yes) = approximately **0.5367**. Rejection sampling gives an estimate that may differ slightly due to randomness; larger sample sizes usually make the estimate closer to the exact value.